# 02 — Phase 3 modelling-data preparation

This notebook creates the documented modelling table for contemporaneous dissolved-oxygen regression. It never overwrites the raw NIEA export.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.preprocessing import (
    WATER_QUALITY_MEASUREMENT_FEATURES,
    load_niea_raw,
    prepare_water_quality_modeling_data,
)

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'niea_river_water_quality_1990_2024.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'water_quality_modeling.csv'
AUDIT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'water_quality_modeling_audit.json'
raw_hash_before = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
raw_data = load_niea_raw(RAW_PATH)
print(f'Raw shape: {raw_data.shape}; SHA-256: {raw_hash_before}')

## Target audit

Rows with missing dissolved oxygen cannot support supervised regression. Qualified target values are censored outcomes and are excluded rather than silently treated as exact.

In [ ]:
print('Missing DO target:', int(raw_data['DO_mg_l_'].isna().sum()))
print('DO qualifiers:')
display(raw_data['Q_DO'].value_counts(dropna=False).to_frame('rows'))
display(raw_data.loc[raw_data['Q_DO'].notna(), ['OBJECTID', 'StationCode', 'Date', 'Q_DO', 'DO_mg_l_']])

## Predictor censoring

Numeric companions are preserved as reported. Separate below-limit and above-limit flags carry qualifier information. No half-limit substitution is applied.

In [ ]:
qualifier_columns = ['Q_BOD', 'Q_NH4N', 'Q_NO2N', 'Q_NO3N', 'Q_PSOL', 'Q_pH']
censoring_summary = pd.DataFrame({
    '< rows': [(raw_data[column] == '<').sum() for column in qualifier_columns],
    '> rows': [(raw_data[column] == '>').sum() for column in qualifier_columns],
    'other nonblank': [
        (raw_data[column].notna() & ~raw_data[column].isin(['<', '>'])).sum()
        for column in qualifier_columns
    ],
}, index=qualifier_columns)
display(censoring_summary)

## Complete cases versus imputation

Complete-case filtering is quantified but not used for the main baseline. Missing predictor values remain in the processed CSV and are median-imputed inside each fitted training pipeline.

In [ ]:
processed_data, audit = prepare_water_quality_modeling_data(raw_data)
display(pd.Series(audit['feature_missing_counts'], name='missing_rows').to_frame())
print(f"Processed rows: {audit['processed_rows']:,}")
print(f"Complete core-feature rows: {audit['complete_case_rows']:,} "
      f"({audit['complete_case_percent_of_processed']:.3f}% retained)")
print('Main strategy: retain eligible rows; fit median imputer on training data only.')

## Extreme values

No IQR-based deletion, clipping or winsorisation is performed. `docs/extreme_value_review.md` records quantiles, flagged counts, stations and dates. Eligible extreme target values remain available for error analysis.

## Transformation record

In [ ]:
for transformation in audit['transformations']:
    print('-', transformation)
print('Output columns:')
print(processed_data.columns.tolist())
display(processed_data.head())

## Save processed modelling data

Station, source-row, date and year metadata are retained for validation and error analysis but are not baseline predictors.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
processed_data.to_csv(OUTPUT_PATH, index=False)
with AUDIT_PATH.open('w', encoding='utf-8') as handle:
    json.dump(audit, handle, indent=2)
raw_hash_after = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
assert raw_hash_before == raw_hash_after, 'Raw dataset was changed.'
assert processed_data['dissolved_oxygen_mg_l'].notna().all()
print(f'Saved {len(processed_data):,} rows to {OUTPUT_PATH}')
print(f'Saved audit to {AUDIT_PATH}')
print('Raw file hash unchanged:', raw_hash_after)

## Cleaning decisions carried forward

- Exclude missing and qualified targets.
- Preserve predictor reported values and add qualifier indicators.
- Preserve missing predictors for train-only imputation.
- Retain statistical outliers because no individual error has been verified.
- Retain station/date metadata for group-aware validation and error analysis.
- Exclude identifiers, station/location/basin and coordinates from the baseline feature matrix.